In [20]:
import torch
from torch.utils.data import Dataset
import os
from PIL import Image
import xmltodict
from torchvision.transforms import ToTensor


# 自定义加载VOC图像的DataLoader
class VOCDataset(Dataset):
    def __init__(
        self, image_folder, label_folder, transform=None, target_transform=None
    ):
        self.image_folder = image_folder
        self.label_folder = label_folder
        self.transform = transform
        self.target_transform = target_transform
        self.img_names = os.listdir(self.image_folder)  # 图像文件名列表
        # print(self.image_folder)  # '../../yolo_data/HelmetDataset-VOC/train/images'
        # print(self.img_names)  # ['new1.png', 'new10.png', 'new100.png', 'new101.png' ...]
        self.class_list = ["no helmet", "motor", "number", "with helmet"]

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, index):
        img_name = self.img_names[index]
        img_path = os.path.join(self.image_folder, img_name)
        print(img_path)
        image = Image.open(img_path)
        print(image)
        label_name = img_name.split(".")[0] + ".xml"
        label_path = os.path.join(self.label_folder, label_name)
        with open(label_path, "r", encoding="utf-8") as f:
            label_content = f.read()  # 读取VOC label数据，为xml格式
            label_dict = xmltodict.parse(label_content)  # 解析xml为字典格式
            print(label_dict)
            print(label_dict["annotation"]["object"])  # 包含所有目标框的列表
            objects = label_dict["annotation"]["object"]
            target = []
            for object in objects:
                class_name = object["name"]  # 目标分类名称
                object_class_id = self.class_list.index(
                    class_name
                )  # 目标分类id（索引）
                bbox = object["bndbox"]
                xmin = float(bbox["xmin"])  # 目标框坐标
                ymin = float(bbox["ymin"])
                xmax = float(bbox["xmax"])
                ymax = float(bbox["ymax"])
                target.extend([object_class_id, xmin, ymin, xmax, ymax])
            print(target)
            if self.transform:
                image = self.transform(image)
        return image, target


if __name__ == "__main__":
    # 测试数据集
    image_folder = "../../yolo_data/HelmetDataset-VOC/train/images"
    label_folder = "../../yolo_data/HelmetDataset-VOC/train/labels"
    dataset = VOCDataset(image_folder, label_folder, transform=ToTensor())
    print(len(dataset))
    print(dataset[0])

97
../../yolo_data/HelmetDataset-VOC/train/images\new1.png
<PIL.PngImagePlugin.PngImageFile image mode=RGB size=283x592 at 0x2264F263130>
{'annotation': {'folder': 'images', 'filename': 'new1.png', 'path': 'C:\\Users\\xiaotudui\\Desktop\\自定义数据集-未包含标注\\train\\images\\new1.png', 'source': {'database': 'Unknown'}, 'size': {'width': '283', 'height': '592', 'depth': '3'}, 'segmented': '0', 'object': [{'name': 'with helmet', 'pose': 'Unspecified', 'truncated': '0', 'difficult': '0', 'bndbox': {'xmin': '115', 'ymin': '10', 'xmax': '202', 'ymax': '94'}}, {'name': 'number', 'pose': 'Unspecified', 'truncated': '0', 'difficult': '0', 'bndbox': {'xmin': '88', 'ymin': '469', 'xmax': '181', 'ymax': '522'}}, {'name': 'motor', 'pose': 'Unspecified', 'truncated': '0', 'difficult': '0', 'bndbox': {'xmin': '26', 'ymin': '276', 'xmax': '267', 'ymax': '573'}}]}}
[{'name': 'with helmet', 'pose': 'Unspecified', 'truncated': '0', 'difficult': '0', 'bndbox': {'xmin': '115', 'ymin': '10', 'xmax': '202', 'ymax':